# Felius source-faithful segmentation audit

Forensic audit of the current periodic-window rule against the released Felius walking-interval procedure. This notebook performs no model fitting and does not read frozen evaluation cohorts.

## What differs from the released pipeline

The Felius `Reliability-of-Gait` implementation trims 200 samples from each acquisition edge, identifies a two-minute walking interval from stationary phases using acceleration and gyroscope magnitude, and operates on three 6-DoF sensors. The current materialisation instead retained a five-second window only if a foot-magnitude periodicity rule passed. This audit reproduces the released *walking-interval selection logic* as closely as the released raw files permit, then intersects the three sensor intervals to preserve synchronized LB/LF/RF windows. Gyroscope bias calibration is not applied because the release does not include the per-device calibration table; this limitation is explicit.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name.lower()=='notebooks' else ROOT
manifest=pd.read_csv(ROOT/'data'/'interim'/'ml_readiness_manifest.csv'); prior=pd.read_csv(ROOT/'data'/'processed'/'validated_window_metadata.csv')
f=manifest[manifest.dataset_id.eq('felius_2024')].copy(); old=prior[prior.dataset_id.eq('felius_2024')].groupby('trial_id').size().rename('current_periodic_windows')
def standphases(acc, gyr, position):
    am=np.linalg.norm(acc,axis=1); gm=np.linalg.norm(gyr,axis=1); ta=am.mean()+am.std(); tg=gm.mean() if position=='foot' else gm.mean()+.2*gm.std(); runs=[]; start=None
    for i,ok in enumerate((am<ta)&(gm<tg)):
        if ok and start is None: start=i
        if start is not None and ((not ok) or i==len(am)-1):
            end=i if not ok else i+1
            if end-start>30: runs.append(np.arange(start,end))
            start=None
    return runs
def source_walk_interval(acc,gyr,position,min_diff,max_diff):
    # Mirrors the released beginEnd/walksig logic, after its 200-sample edge trim.
    acc,gyr=acc[200:-200],gyr[200:-200]; phases=standphases(acc,gyr,position)
    if len(phases)<50: return None, len(phases)
    begins=phases[:25]; ends=phases[-25:]
    for s in sorted(begins,key=len,reverse=True):
        for e in sorted(ends,key=len,reverse=True):
            diff=e[0]-s[-1]
            if min_diff < diff < max_diff:
                return (max(0,s[-1]-100)+200,min(len(acc),e[0]+100)+200),len(phases)
    return None,len(phases)
rows=[]
for row in f.itertuples(index=False):
    paths={Path(p).name.rsplit('_',1)[-1].replace('.csv',''):ROOT/p for p in row.raw_files.split('|')}
    intervals=[]; phase_counts=[]; bad=False
    for key,pos,lo,hi in [('leftfoot','foot',10000,13000),('rightfoot','foot',10000,13000),('lowback','lowback',10000,13500)]:
        q=pd.read_csv(paths[key]); vals=q[['ax','ay','az','gx','gy','gz']].to_numpy(float); interval,nphase=source_walk_interval(vals[:,:3],np.deg2rad(vals[:,3:]),pos,lo,hi); intervals.append(interval); phase_counts.append(nphase); bad |= interval is None
    if bad: start=end=np.nan; new_windows=0; status='no_common_source_interval'
    else:
        start=max(i[0] for i in intervals); end=min(i[1] for i in intervals); new_windows=max(0,(end-start-500)//250+1); status='included' if new_windows else 'common_interval_too_short'
    rows.append({'trial_id':row.trial_id,'participant_key':f'felius_2024:{row.subject}','label':row.label,'current_periodic_windows':int(old.get(row.trial_id,0)),'source_walk_start':start,'source_walk_end':end,'source_walk_samples':0 if pd.isna(start) else int(end-start),'source_walk_windows_2p5s_hop':int(new_windows),'stationary_phases_min':min(phase_counts),'status':status})
audit=pd.DataFrame(rows); out=ROOT/'data'/'interim'/'felius_source_faithful_segmentation_audit.csv'; audit.to_csv(out,index=False)
print(audit.groupby(['label','status']).size()); print(audit.groupby('label')[['current_periodic_windows','source_walk_windows_2p5s_hop','source_walk_samples']].agg(['count','mean','median','sum']).round(2)); print('wrote',out)

C:\Users\frank\AppData\Local\Temp\ipykernel_18180\1292847477.py:7: RuntimeWarning: Mean of empty slice
  am=np.linalg.norm(acc,axis=1); gm=np.linalg.norm(gyr,axis=1); ta=am.mean()+am.std(); tg=gm.mean() if position=='foot' else gm.mean()+.2*gm.std(); runs=[]; start=None
C:\Users\frank\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\frank\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\frank\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\_core\_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\frank\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\_core\_methods.py:211: RuntimeWa

label    status                   
healthy  included                      53
         no_common_source_interval      6
stroke   included                     258
         no_common_source_interval     60
dtype: int64
        current_periodic_windows                       \
                           count   mean median    sum   
label                                                   
healthy                       59  49.51   50.0   2921   
stroke                       318  42.28   48.0  13445   

        source_walk_windows_2p5s_hop                      source_walk_samples  \
                               count   mean median    sum               count   
label                                                                           
healthy                           59  38.86   43.0   2293                  59   
stroke                           318  36.91   45.0  11738                 318   

                                     
             mean   median      sum  
label           

C:\Users\frank\AppData\Local\Temp\ipykernel_18180\1292847477.py:7: RuntimeWarning: Mean of empty slice
  am=np.linalg.norm(acc,axis=1); gm=np.linalg.norm(gyr,axis=1); ta=am.mean()+am.std(); tg=gm.mean() if position=='foot' else gm.mean()+.2*gm.std(); runs=[]; start=None
C:\Users\frank\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
C:\Users\frank\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\_core\_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\frank\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\_core\_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
C:\Users\frank\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\_core\_methods.py:211: RuntimeWa

A corrected Felius representation may proceed only if the audit confirms a usable common walking interval for both labels and removes the observed label-asymmetric periodicity exclusion. Any corrected materialisation will be a new, versioned training representation; the current baseline and frozen cohorts will not be overwritten or re-used for selection.